In [2]:
%pip install datasets matplotlib scikit-learn
from datasets import load_dataset

ds = load_dataset("rajistics/indian_food_images")


Note: you may need to restart the kernel to use updated packages.


/Users/muskangoyal/miniforge3/envs/food_env_fix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [4]:
ds=ds["train"].train_test_split(test_size=0.2,seed=42)
val_test=ds["test"].train_test_split(test_size=0.5, seed=42)

train_ds=ds["train"]
val_ds=val_test["train"]
test_ds=val_test["test"]

In [5]:
IMG_SIZE = 224
BATCH_SIZE=32

def preprocess(example):
    image=example["image"]
    image=image.resize((IMG_SIZE, IMG_SIZE))
    image=np.array(image, stype=np.float32)/255.0
    label=example["label"]
    return image, label


In [6]:
def to_tf_dataset(hf_ds, shuffle=True, is_training=True):
    def generator():
        for example in hf_ds:
            img = example["image"]
            # Ensure image is in RGB (converts RGBA or Grayscale automatically)
            img = img.convert("RGB") 
            yield np.array(img), example["label"]

    tf_ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            # We use None for height/width because images might be different sizes
            tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8), 
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )

    def preprocess(image, label):
        # Resize and scale
        image = tf.image.resize(image, (224, 224))
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    tf_ds = tf_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        tf_ds = tf_ds.shuffle(1000)
    
    if is_training:
        tf_ds = tf_ds.repeat()

    return tf_ds.batch(32).prefetch(tf.data.AUTOTUNE)

# Re-initialize your datasets
train_tf = to_tf_dataset(train_ds, shuffle=True, is_training=True)
val_tf   = to_tf_dataset(val_ds, shuffle=False, is_training=False) # No repeat
test_tf  = to_tf_dataset(test_ds, shuffle=False, is_training=False) # No repeat

# Calculate steps accurately so the progress bar moves
steps_per_epoch = len(train_ds) // 32
val_steps = len(val_ds) // 32

2026-02-19 23:14:03.270263: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-19 23:14:03.270294: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-19 23:14:03.270301: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-02-19 23:14:03.270331: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-19 23:14:03.270340: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])


In [8]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False


In [9]:
num_classes = len(set(train_ds["label"]))


In [10]:
inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)


In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [12]:
history = model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=15,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps
)


Epoch 1/15


2026-02-19 23:14:04.821920: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_tf:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred))


In [ ]:
model.save("indian_food_classifier.h5")


In [ ]:
import json
from datasets import load_dataset

dataset = load_dataset("rajistics/indian_food_images")

# Get label names
label_names = dataset["train"].features["label"].names

# Save to json
with open("label_map.json", "w") as f:
    json.dump(label_names, f)

print("label_map.json created ✅")
